In [17]:
# Loading 
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from sympy import symbols, diff, solve, lambdify
from sympy.physics.quantum.tensorproduct import TensorProduct
# Define the variable and the function
ri = symbols('ri')
phi = symbols('phi')
lbd = symbols('lbd')
R = symbols('R')
p = symbols('p')


In [18]:
# Initialize the parameters
L = 0.05
R_i = 0.00035
R_o = 0.0023
Theta = 0
alpha = 1.44
mu_1 = 427560
mu_2 = 8500000
P = np.linspace(0, 1000000, 100)  # Pressure range from 0 to 100 kPa

In [19]:
# Define the function f(ri, phi, R, lambda) using sympy
# Deformation gradient tensor
r = sp.sqrt((R**2-R_i**2)/lbd + ri**2)
F = sp.Matrix([[R/(r*lbd), 0, 0], [0, r/R, r*phi/L], [0, 0, lbd]])
S = sp.Matrix([[0], [sp.sin(alpha)], [sp.cos(alpha)]])
print("Deformation Gradient Tensor F:")
sp.pprint(F)
print("\nDirection Vector s:")
sp.pprint(S)
# Calculate the Cauchy-Green deformation tensor C
s = F * S
sp.pprint(s)

Deformation Gradient Tensor F:
⎡              R                                                               ↪
⎢──────────────────────────────              0                               0 ↪
⎢         _____________________                                                ↪
⎢        ╱        2                                                            ↪
⎢       ╱    2   R  - 1.225e-7                                                 ↪
⎢lbd⋅  ╱   ri  + ─────────────                                                 ↪
⎢    ╲╱               lbd                                                      ↪
⎢                                                                              ↪
⎢                                     _____________________                    ↪
⎢                                    ╱        2                                ↪
⎢                                   ╱    2   R  - 1.225e-7               _____ ↪
⎢                                  ╱   ri  + ─────────────              ╱     

In [41]:
# First and fourth invariants of C
I1 = sp.trace(sp.MatMul(F, F.T))
I4 = sp.MatMul(s.T, s)
print(sp.shape(I4))   
I4 = I4[0,0]
sp.pprint(I4)

(1, 1)
                                                                               ↪
                          ⎛                                                    ↪
                          ⎜                                                    ↪
                          ⎜                        _____________________       ↪
                          ⎜                       ╱        2               0.9 ↪
                      2   ⎜                      ╱    2   R  - 1.225e-7        ↪
0.0170103438010126⋅lbd  + ⎜2.60847417476291⋅φ⋅  ╱   ri  + ─────────────  + ─── ↪
                          ⎝                   ╲╱               lbd             ↪

↪                                           2
↪                     _____________________⎞ 
↪                    ╱        2            ⎟ 
↪                   ╱    2   R  - 1.225e-7 ⎟ 
↪ 91458348191686⋅  ╱   ri  + ───────────── ⎟ 
↪                ╲╱               lbd      ⎟ 
↪ ─────────────────────────────────────────⎟ 
↪                   R 

In [45]:
sigma_tube = mu_1 * sp.MatMul(F, F.T) 
sigma_coil = mu_2 * TensorProduct(s, s.T) * (sp.sqrt(I4) - 1) / sp.sqrt(I4) 
sigma_total = sigma_tube + sigma_coil - (p * sp.eye(3))
sp.shape(sigma_total)

sp.pprint(sigma_total[0,0])

                2             
        427560⋅R              
────────────────────────── - p
     ⎛       2           ⎞    
   2 ⎜  2   R  - 1.225e-7⎟    
lbd ⋅⎜ri  + ─────────────⎟    
     ⎝           lbd     ⎠    


In [ ]:
f_pressure = R / (R**2 - R_i**2 + lbd*ri**2) * (sigma_total[1,1] - sigma_total[0,0])
f_load = np.pi * R/lbd * (2 * sigma_total[2,2] - sigma_total[0,0] - sigma_total[1,1])
f_moment = sigma_total[1,2] * r * R / lbd * 2 * np.pi

In [48]:
# analytical integral for pressure and load and moment
f_pressure_integral = sp.integrate(f_pressure, (R, R_i, R_o))
f_load_integral = sp.integrate(f_load, (R, R_i, R_o))
f_moment_integral = sp.integrate(f_moment, (R, R_i, R_o))

KeyboardInterrupt: 

In [50]:
sp.pprint(f_pressure_integral)

                  ⎛                                                            ↪
                  ⎜                                                            ↪
                  ⎜                                                            ↪
                  ⎜                                                            ↪
                  ⎜    0.0023                                                  ↪
                  ⎜      ⌠                                                     ↪
                  ⎜      ⎮                                                     ↪
-27563327.6630735⋅⎜1.0⋅  ⎮     ─────────────────────────────────────────────── ↪
                  ⎜      ⎮          __________________________________________ ↪
                  ⎜      ⎮         ╱      4  2                      3          ↪
                  ⎜      ⎮        ╱  1.0⋅R ⋅φ    0.760182606202572⋅R ⋅φ        ↪
                  ⎜      ⎮       ╱   ───────── + ────────────────────── + 0.00 ↪
                  ⎜      ⎮  